3. Naloga - gRPC
V gRPC napiˇsite naslednji program: uporabnik na streˇznik poˇsilja toˇcke (x, y, z),
streˇznik izraˇcuna razdaljo toˇcke od izhodiˇsˇca po formuli r =
p
(x
2 + y
2 + z
2
), nato pa
razdaljo poˇslje nazaj uporabniku.
Vaˇsa .proto datoteka naj vsebuje message Tocka, ki vsebuje tri floate x, y, z, in
message Razdalja, ki vsebuje float r.
Vaˇs service naj vsebuje dva rpc. Prvi sprejme Tocka in vrne Razdalja (uporabnik poˇslje
toˇcko, sprejme razdaljo), drugi pa prejme stream Tocka in vrne stream Razdalja
(uporabnik poˇslje veˇc toˇck, dobi veˇc razdalj).
Naloga: Napiˇsite .proto datoteko, program za streˇznik in program za uporabnika.

In [ ]:
syntax = "proto3";

package grpc;

message Tocka {
    float x = 1;
    float y = 2;
    float z = 3;
}

message Razdalja {
    float r = 1;
}

service RazdaljaService {

    // ena točka -> ena razdalja
    rpc PosljiTocko (Tocka) returns (Razdalja);

    // stream točk -> stream razdalj
    rpc PosljiVecTock (stream Tocka) returns (stream Razdalja);
}

In [ ]:
pip install grpcio grpcio-tools

In [ ]:
python -m grpc_tools.protoc -I. --python_out=. --grpc_python_out=. razdalja.proto

In [ ]:
from concurrent import futures
import math
import grpc

import razdalja_pb2 as pb2
import razdalja_pb2_grpc as pb2_grpc


class Service(pb2_grpc.RazdaljaServiceServicer):

    # unary -> unary
    def PosljiTocko(self, request, context):

        x = request.x
        y = request.y
        z = request.z

        r = math.sqrt(x*x + y*y + z*z)

        print(f"Prejel: ({x}, {y}, {z}) -> {r}")

        return pb2.Razdalja(r=r)

    # stream -> stream
    def PosljiVecTock(self, request_iterator, context):

        for point in request_iterator:

            x = point.x
            y = point.y
            z = point.z

            r = math.sqrt(x*x + y*y + z*z)

            print(f"STREAM: ({x}, {y}, {z}) -> {r}")

            yield pb2.Razdalja(r=r)


server = grpc.server(futures.ThreadPoolExecutor(max_workers=10))

pb2_grpc.add_RazdaljaServiceServicer_to_server(
    Service(),
    server
)

server.add_insecure_port('[::]:50051')

server.start()

print("Server teče na portu 50051")

server.wait_for_termination()

In [ ]:
import grpc

import razdalja_pb2 as pb2
import razdalja_pb2_grpc as pb2_grpc


channel = grpc.insecure_channel('localhost:50051')

stub = pb2_grpc.RazdaljaServiceStub(channel)


# =========================
# ENA TOČKA
# =========================

point = pb2.Tocka(x=3, y=4, z=5)

response = stub.PosljiTocko(point)

print("Razdalja:", response.r)


# =========================
# STREAM TOČK
# =========================

def generator():

    points = [
        pb2.Tocka(x=1, y=1, z=1),
        pb2.Tocka(x=2, y=2, z=2),
        pb2.Tocka(x=3, y=3, z=3),
    ]

    for p in points:
        yield p


responses = stub.PosljiVecTock(generator())

for r in responses:
    print("Stream razdalja:", r.r)